# PART 03 · 4장 회귀 모델링

교재 260~278쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

회귀에서 사용할 라이브러리를 한 번에 불러온다.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

배달 시간 데이터를 불러와 크기와 결측치를 확인해본다.

In [2]:
base = 'data/part03_type2/regression_delivery_minutes'
train = pd.read_csv(base+'/train.csv')
test = pd.read_csv(base+'/test.csv')
submit = pd.read_csv(base+'/sample_submission.csv')
print(train.shape)
print(test.shape)
print(submit.shape)
print(set(train.columns)-set(test.columns))
print(train[['rain_mm', 'traffic_index']].isnull().sum())

(840, 15)
(280, 14)
(280, 1)
{'delivery_minutes'}
rain_mm          42
traffic_index    21
dtype: int64


## 01 회귀 문제 처리

### 1 회귀 문제의 특징

#### 1. 분류와 회귀의 차이

#### 2. target 확인

target인 delivery_minutes의 분포를 확인해본다.

In [3]:
print(train['delivery_minutes'].describe())

count    840.000000
mean      53.337738
std       13.600286
min       21.600000
25%       43.600000
50%       52.900000
75%       61.200000
max      100.500000
Name: delivery_minutes, dtype: float64


### 2 풀이 흐름과 데이터 준비

#### 1. 회귀 문제의 전체 흐름

#### 2. 공통 데이터 준비 1

target을 분리하고 학습용과 검증용으로 나누는 준비 코드부터 실행한다.

In [4]:
target = 'delivery_minutes'
X_raw = train.drop(columns=[target, 'order_id']).copy()
y = train[target].copy()
X_test_raw = test.drop(columns=['order_id']).copy()
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=0
)
num_cols = X_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_train_raw[col].median()
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)
    X_test_raw[col] = X_test_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_train_raw[col].mode()[0]
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)
    X_test_raw[col] = X_test_raw[col].fillna(fill_value)

#### 3. 공통 데이터 준비 2

인코딩과 컬럼 맞추기까지 마쳐 회귀 입력을 완성한다.

In [5]:
X_train = pd.get_dummies(X_train_raw)
X_val = pd.get_dummies(X_val_raw).reindex(
    columns=X_train.columns, fill_value=0
)
X_test = pd.get_dummies(X_test_raw).reindex(
    columns=X_train.columns, fill_value=0
)
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape)
print(X_train.isnull().sum().sum())
print(X_val.isnull().sum().sum())
print(X_test.isnull().sum().sum())

(672, 28) (168, 28) (280, 28)
(672,) (168,)
0
0
0


## 02 LinearRegression

### 1 모델 학습과 예측

#### 1. 모델 학습과 검증 예측

선형회귀 모델을 학습하고 검증 데이터로 네 가지 지표를 계산해본다.

In [6]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_val)
linear_mae = mean_absolute_error(y_val, linear_pred)
linear_mse = mean_squared_error(y_val, linear_pred)
linear_rmse = linear_mse**0.5
linear_r2 = r2_score(y_val, linear_pred)
print('MAE: ', round(linear_mae, 4))
print('MSE: ', round(linear_mse, 4))
print('RMSE: ', round(linear_rmse, 4))
print('R2: ', round(linear_r2, 4))

MAE:  6.0664
MSE:  58.057
RMSE:  7.6195
R2:  0.6138


#### 2. 실젯값과 예측값 확인

실젯값과 예측값을 나란히 놓고 비교해본다.

In [7]:
linear_check = pd.DataFrame({
    'actual': y_val.head(5).values,
    'pred': linear_pred[:5]
})
print(linear_check)

   actual       pred
0    66.3  64.443854
1    45.5  38.398392
2    46.9  52.131047
3    61.2  66.267676
4    48.9  43.739370


### 2 회귀계수 확인

#### 1. 회귀계수 확인

회귀계수를 절댓값 기준으로 정렬해 영향이 큰 변수를 확인해본다.

In [8]:
coef = pd.Series(linear_model.coef_, index=X_train.columns)
coef_order = coef.abs().sort_values(ascending=False)
print('intercept: ', round(linear_model.intercept_, 4))
print(coef.loc[coef_order.head(6).index].round(4))

intercept:  25.7352
distance_km       4.7868
store_type_피자     4.7659
promotion         4.6807
store_type_카페    -3.9855
store_type_샐러드   -2.2946
store_type_치킨     2.2529
dtype: float64


## 03 DecisionTreeRegressor

### 1 학습과 변수 중요도

#### 1. 모델 학습

max_depth를 5로 제한한 회귀 트리를 학습하고 검증 지표를 확인해본다.

In [9]:
tree_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=0
)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_val)
tree_mae = mean_absolute_error(y_val, tree_pred)
tree_mse = mean_squared_error(y_val, tree_pred)
tree_rmse = tree_mse**0.5
tree_r2 = r2_score(y_val, tree_pred)
print('MAE: ', round(tree_mae, 4))
print('MSE: ', round(tree_mse, 4))
print('RMSE: ', round(tree_rmse, 4))
print('R2: ', round(tree_r2, 4))
print('depth: ', tree_model.get_depth())
print('leaves: ', tree_model.get_n_leaves())

MAE:  8.653
MSE:  115.9268
RMSE:  10.7669
R2:  0.2288
depth:  5
leaves:  30


#### 2. 변수 중요도 확인

feature_importances_로 변수 중요도를 확인해본다.

In [10]:
tree_importance = pd.Series(
    tree_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)
print(tree_importance.head(6).round(4))

distance_km        0.5740
traffic_index      0.2335
item_count         0.0527
rider_available    0.0515
request_length     0.0300
rain_mm            0.0234
dtype: float64


## 04 RandomForestRegressor

### 1 모델 학습

랜덤포레스트 회귀 모델을 학습하고 같은 지표를 확인해본다.

In [11]:
forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=0
)
forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_val)
forest_mae = mean_absolute_error(y_val, forest_pred)
forest_mse = mean_squared_error(y_val, forest_pred)
forest_rmse = forest_mse**0.5
forest_r2 = r2_score(y_val, forest_pred)
print('MAE: ', round(forest_mae, 4))
print('MSE: ', round(forest_mse, 4))
print('RMSE: ', round(forest_rmse, 4))
print('R2: ', round(forest_r2, 4))

MAE:  7.0549
MSE:  74.0563
RMSE:  8.6056
R2:  0.5073


### 2 예측값과 변수 중요도

예측값 일부와 변수 중요도를 확인해본다.

In [12]:
print(pd.Series(forest_pred[:5]).round(3).tolist())
forest_importance = pd.Series(
    forest_model.feature_importances_,
    index=X_train.columns).sort_values(ascending=False)
print(forest_importance.head(6).round(4))

[64.687, 35.341, 53.484, 52.891, 44.658]
distance_km        0.4254
traffic_index      0.1634
rider_available    0.0996
item_count         0.0547
rain_mm            0.0508
request_length     0.0477
dtype: float64


## 05 예측값 생성

### 1 예측값 만들기

#### 1. 검증 예측과 test 예측

#### 2. 전체 train으로 다시 학습하기

검증 모델은 train의 일부만 학습에 사용했다. 최종 test 예측을 만들 때는 선택한 모델을 전체 X와 y로 다시 학습해 제공된 학습 데이터를 모두 활용한다.

In [13]:
X_full = train.drop(columns=[target, 'order_id']).copy()
X_test_full = test.drop(columns=['order_id']).copy()
num_cols = X_full.select_dtypes(exclude='object').columns
cat_cols = X_full.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_full[col].median()
    X_full[col] = X_full[col].fillna(fill_value)
    X_test_full[col] = X_test_full[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_full[col].mode()[0]
    X_full[col] = X_full[col].fillna(fill_value)
    X_test_full[col] = X_test_full[col].fillna(fill_value)
X_full = pd.get_dummies(X_full)
X_test_full = pd.get_dummies(X_test_full).reindex(
    columns=X_full.columns,
    fill_value=0
)
final_model = LinearRegression()
final_model.fit(X_full, y)
test_pred = final_model.predict(X_test_full)
result = pd.DataFrame({'pred': test_pred})
print(result.shape)
print(result.head())
print(
    round(result['pred'].min(), 3),
    round(result['pred'].max(), 3),
    round(result['pred'].mean(), 3)
)

(280, 1)
        pred
0  62.716764
1  61.246269
2  47.857509
3  42.684609
4  42.567160
24.943 86.234 52.895


### 2 예측값 범위 확인

#### 1. 예측값 범위 확인

## 06 MAE

### 1 MAE 계산

#### 1. MAE 계산

mean_absolute_error( )로 MAE를 계산해본다.

In [14]:
mae = mean_absolute_error(y_val, linear_pred)
print(round(mae, 4))

6.0664


#### 2. 직접 계산하기

같은 값을 공식대로 직접 계산해 함수 결과와 비교해본다.

In [15]:
direct_mae = abs(y_val.values - linear_pred).mean()
print(round(direct_mae, 4))

6.0664


### 2 절대오차 확인

#### 1. 일부 행의 절대오차

## 07 MSE

### 1 MSE 계산

#### 1. MSE 계산

mean_squared_error( )로 MSE를 계산해본다.

In [16]:
mse = mean_squared_error(y_val, linear_pred)
print(round(mse, 4))

58.057


#### 2. 직접 계산하기

오차를 제곱해 평균 내는 공식대로 직접 계산해본다.

In [17]:
direct_mse = ((y_val.values - linear_pred) ** 2).mean()
print(round(direct_mse, 4))

58.057


### 2 MAE와 비교

## 08 RMSE

### 1 RMSE 계산

MSE에 제곱근을 적용하면 RMSE가 된다.

In [18]:
rmse = mean_squared_error(y_val, linear_pred) ** 0.5
print(round(rmse, 4))

7.6195


### 2 MSE와 RMSE의 관계

## 09 결정계수(R²)

### 1 R² 계산

#### 1. R² 계산

r2_score( )로 R²를 계산해본다.

In [19]:
r2 = r2_score(y_val, linear_pred)
print(round(r2, 4))

0.6138


#### 2. model.score( )와 비교

회귀 모델의 score( ) 메서드도 같은 R²를 반환한다. 두 결과를 비교해본다.

In [20]:
print(round(linear_model.score(X_val, y_val), 4))
print(round(r2_score(y_val, linear_pred), 4))

0.6138
0.6138


### 2 지표의 방향 구분

#### 1. 지표의 방향 구분

## 10 회귀 결과 확인

### 1 예측 결과와 지표 확인

#### 1. 실젯값, 예측값, 오차 확인

실젯값, 예측값, 오차, 절대오차를 한 표에서 확인해본다.

In [21]:
result_check = pd.DataFrame({
    'actual': y_val.values,
    'pred': linear_pred
})
result_check['error'] = (
    result_check['actual'] - result_check['pred']
)
result_check['absolute_error'] = result_check['error'].abs()
print(result_check.head().round(3))

   actual    pred  error  absolute_error
0    66.3  64.444  1.856           1.856
1    45.5  38.398  7.102           7.102
2    46.9  52.131 -5.231           5.231
3    61.2  66.268 -5.068           5.068
4    48.9  43.739  5.161           5.161


#### 2. 네 가지 평가지표 모아 보기

### 2 모델 비교와 최종 점검

#### 1. 모델별 검증 결과 비교

#### 2. 최종 예측값 점검

검증 결과가 가장 좋은 모델을 전체 train으로 다시 학습해 최종 예측값을 점검해본다.

In [22]:
final_model = LinearRegression()
final_model.fit(X_full, y)
result = pd.DataFrame({
    'pred': final_model.predict(X_test_full)
})
print(result.shape)
print(result.columns.tolist())
print(result.isnull().sum())
print(result['pred'].describe())

(280, 1)
['pred']
pred    0
dtype: int64
count    280.000000
mean      52.894644
std       10.705237
min       24.942784
25%       45.934385
50%       52.520533
75%       59.774209
max       86.233988
Name: pred, dtype: float64


결과 데이터프레임은 280행 1열이고 컬럼명은 pred다. 결측치는 없으며 예측값 범위도 train target 범위인 21.6~100.5에서 크게 벗어나지 않는다. 파일 저장과 제출 전 검증 절차는 다음 장에서 자세히 다룬다.

> 설명용 코드입니다. 그대로 실행되지 않습니다.

```python
mae = mean_absolute_error(y_val, pred)  # 작을수록 좋음
mse = mean_squared_error(y_val, pred)  # 작을수록 좋음
rmse = mse ** 0.5 # 작을수록 좋음
r2 = r2_score(y_val, pred) # 클수록 좋음
```